In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(
    "../processed_data/eda_cleaned.csv"
)

In [3]:
df.head(5).T

,0,1,2,3,4
year,2025,2025,2025,2025,2025
month,1,1,1,1,1
carrier,G4,G4,G4,G4,G4
carrier_name,Allegiant Air,Allegiant Air,Allegiant Air,Allegiant Air,Allegiant Air
airport,ELM,ELP,EUG,EVV,EWR
airport_name,"Elmira/Corning, NY: Elmira/Corning Regional","El Paso, TX: El Paso International","Eugene, OR: Mahlon Sweet Field","Evansville, IN: Evansville Regional","Newark, NJ: Newark Liberty International"
arr_flights,30.0,2.0,28.0,18.0,31.0
arr_del15,0.0,0.0,8.0,1.0,5.0
carrier_ct,0.0,0.0,3.74,0.0,2.17
weather_ct,0.0,0.0,0.0,1.0,0.0


In [4]:
df = df.drop(columns=['carrier','airport'])

In [5]:
df.shape

(397283, 20)

WE WILL AGAIN CREATE THE 3 NEW FEATURES

In [6]:
df['cancel_rate'] = (df['arr_cancelled']/df['arr_flights'])

In [7]:
df['divert_rate'] = df['arr_diverted']/df['arr_flights']

In [8]:
df['peak_delay'] = df['month'].isin([6,7,12]).astype(int)

In [9]:
df.sample(13).T

,199590,334465,267434,132982,280551,144270,185136,376343,247141,48857,209528,266561,322812
year,2015,2007,2010,2019,2010,2018,2016,2004,2012,2022,2014,2010,2007
month,4,2,10,4,1,11,5,9,1,12,7,10,9
carrier_name,JetBlue Airways,Northwest Airlines Inc.,Southwest Airlines Co.,Air Wisconsin Airlines Corp,Southwest Airlines Co.,Republic Airline,ExpressJet Airlines Inc.,Delta Air Lines Inc.,American Eagle Airlines Inc.,Republic Airline,ExpressJet Airlines Inc.,Delta Air Lines Inc.,Atlantic Southeast Airlines
airport_name,"Phoenix, AZ: Phoenix Sky Harbor International","Charlotte Amalie, VI: Cyril E King","San Jose, CA: Norman Y. Mineta San Jose Intern...","Grand Rapids, MI: Gerald R. Ford International","Santa Ana, CA: John Wayne Airport-Orange County","Miami, FL: Miami International","Dayton, OH: James M Cox/Dayton International","Huntsville, AL: Huntsville International-Carl ...","Louisville, KY: Louisville Muhammad Ali Intern...","Indianapolis, IN: Indianapolis International","Lubbock, TX: Lubbock Preston Smith International","Washington, DC: Washington Dulles International","Knoxville, TN: McGhee Tyson"
arr_flights,60.0,4.0,1919.0,88.0,1310.0,882.0,148.0,111.0,93.0,750.0,178.0,261.0,112.0
arr_del15,16.0,3.0,413.0,13.0,181.0,114.0,16.0,20.0,19.0,87.0,44.0,64.0,25.0
carrier_ct,3.88,0.96,109.67,2.78,44.47,32.72,2.66,9.23,5.3,27.6,10.56,21.39,15.46
weather_ct,0.0,0.0,5.43,0.0,9.99,1.09,1.6,0.0,0.9,0.47,1.35,1.33,4.98
nas_ct,6.97,2.04,31.81,4.75,22.5,55.89,3.62,4.74,7.44,23.25,11.24,23.91,2.56
security_ct,0.0,0.0,1.54,0.0,1.68,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
df_model = df.drop(columns=['carrier_delay', 'weather_delay', 'nas_delay',
        'security_delay', 'late_aircraft_delay','arr_delay','delay_rate', 'carrier_ct', 
        'weather_ct', 'nas_ct', 'security_ct','arr_del15',
       'late_aircraft_ct', 'arr_cancelled', 'arr_diverted', 'cancel_rate',
       'divert_rate'])

In [11]:
x = df_model
y = df['arr_delay']

In [12]:
x.columns

Index(['year', 'month', 'carrier_name', 'airport_name', 'arr_flights',
       'peak_delay'],
      dtype='object')

In [13]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ohe  = OneHotEncoder(
    #drop= "first",
    handle_unknown= "ignore"
)

In [15]:
transformer = ColumnTransformer(
    transformers=[
        ('cat',
        ohe,
        ['carrier_name', 'airport_name']
        )
    ],
    remainder='passthrough')

In [16]:
x_train_encoded = transformer.fit_transform(x_train)
x_test_encoded = transformer.transform(x_test)

In [17]:
print(x_train_encoded.shape)
print(x_test_encoded.shape)

(278098, 499)
(119185, 499)


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

pipe_lr = Pipeline([("preprocessor", transformer), ("model",LinearRegression())])
pipe_dtr = Pipeline([("preprocessor", transformer), ("model",DecisionTreeRegressor(random_state=42))])
pipe_rfr = Pipeline([("preprocessor", transformer), ("model",RandomForestRegressor(random_state=42))])

In [22]:
pipe_dtr.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['year','month','carrier_name','airport_name','arr_flights','peak_delay']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenate

In [41]:
y_dtr_train = pipe_dtr.predict(x_train)
y_dtr_test = pipe_dtr.predict(x_test)

c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [19]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [43]:

mae_test = mean_absolute_error(y_test, y_dtr_test)
mae_train = mean_absolute_error(y_train, y_dtr_train)
rmse_test = np.sqrt(mean_squared_error(y_test, y_dtr_test))
rmse_train = np.sqrt(mean_squared_error(y_train, y_dtr_train))

r2_test = r2_score(y_test,y_dtr_test)
r2_train = r2_score(y_train, y_dtr_train)

In [44]:
print("MAE for test: ",mae_test)
print("MAE for train: ",mae_train)
print("RMSE for test: ",rmse_test)
print("RMSE for train: ",rmse_train)
print("r2 for test: ",r2_test)
print("r2 for train: ",r2_train)

MAE for test:  1510.004270671645
MAE for train:  0.0
RMSE for test:  5478.368515756969
RMSE for train:  0.0
r2 for test:  0.8139917476144901
r2 for train:  1.0


The unrestricted Decision Tree achieved a perfect fit on the training data (R² = 1.0), indicating that it memorized the training observations. Although this is evidence of overfitting, the test R² remained high (0.8134), suggesting that the model still generalized well to unseen data. Therefore, overfitting was present but did not severely degrade predictive performance. Nevertheless, tree complexity was further investigated using max_depth to determine whether a simpler tree could achieve equal or better generalization.

In [45]:
result = []

for depth in [3,5,7,10,15,20]:
    pipe = Pipeline(
        [
            ("preprocessor", transformer),
            ("model", DecisionTreeRegressor(random_state=42, max_depth=depth))
        ]
    )

    pipe.fit(x_train,y_train)

    y_pred_test = pipe.predict(x_test)
    y_pred_train = pipe.predict(x_train)


    result.append(
        {
            "max depth" : depth,
            "r2 train" : r2_score(y_train, y_pred_train),
            "r2_test" : r2_score(y_test, y_pred_test),
            "train mae" : mean_absolute_error(y_train, y_pred_train),
            "test mae" : mean_absolute_error(y_test, y_pred_test),
            "train rmse" : np.sqrt(mean_squared_error(y_train, y_pred_train)),
            "test rmse" : np.sqrt(mean_squared_error(y_test, y_pred_test))
        }
    )

c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. 

In [46]:
pd.DataFrame(result)

,max depth,r2 train,r2_test,train mae,test mae,train rmse,test rmse
0,3,0.753129,0.751827,2095.647735,2120.329013,6352.287667,6327.946476
1,5,0.810463,0.795915,1742.439233,1772.184688,5565.997335,5738.405033
2,7,0.849182,0.824122,1590.126144,1636.184637,4965.037385,5327.096342
3,10,0.899960,0.837618,1388.885667,1525.485745,4043.740147,5118.641057
4,15,0.954085,0.827231,1033.351185,1459.122354,2739.520236,5279.801787
5,20,0.979638,0.825177,728.823092,1438.653819,1824.331035,5311.104734


The maximum tree depth was tuned to study the effect of model complexity. As depth increased, the training performance improved monotonically, indicating greater model flexibility. However, the test performance improved only up to a depth of 10 and then began to decline slightly, suggesting the onset of overfitting. Therefore, a maximum depth of 10 was selected as it provided the best trade-off between fitting the training data and generalizing to unseen observations.

NOW I SET THE MAX DEPTH TO 10

In [21]:
pipe_dtr = Pipeline([
    ("prerpocessor", transformer),
    ("model", DecisionTreeRegressor(random_state=42, max_depth=10))
    ])

In [35]:
pipe_dtr.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prerpocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['year','month','carrier_name','airport_name','arr_flights','peak_delay']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenate

In [20]:
from sklearn.model_selection import KFold, cross_val_score

In [21]:

kfold = KFold(
    n_splits=5,
    shuffle= True,
    random_state= 42
)

In [ ]:
cv_score = cross_val_score(
    pipe_dtr,
    x_train,
    y_train,
    cv=kfold,
    scoring = "r2",
    n_jobs=1
)

In [39]:
print("fold r2 scores: ", cv_score)
print("mean cv r2: ", cv_score.mean())
print("std of fold scores: ", cv_score.std())

fold r2 scores:  [0.83163014 0.8216913  0.81971242 0.81525147 0.82960614]
mean cv r2:  0.8235782942942654
std of fold scores:  0.006148314644976392


MOVING TOWARDS RANDOM FOREST

In [24]:
pipe_rfr = Pipeline([("preprocessor", transformer),
                    ("model",RandomForestRegressor(random_state=42,n_jobs=-1))])

In [24]:
pipe_rfr.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['year','month','carrier_name','airport_name','arr_flights','peak_delay']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenate

In [25]:
y_rf_train = pipe_rfr.predict(x_train)
y_rf_test = pipe_rfr.predict(x_test)

c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [28]:

mae_test = mean_absolute_error(y_test, y_rf_test)
mae_train = mean_absolute_error(y_train, y_rf_train)
rmse_test = np.sqrt(mean_squared_error(y_test, y_rf_test))
rmse_train = np.sqrt(mean_squared_error(y_train, y_rf_train))

r2_test = r2_score(y_test,y_rf_test)
r2_train = r2_score(y_train, y_rf_train)

In [29]:
print("MAE for test: ",mae_test)
print("MAE for train: ",mae_train)
print("RMSE for test: ",rmse_test)
print("RMSE for train: ",rmse_train)
print("r2 for test: ",r2_test)
print("r2 for train: ",r2_train)

MAE for test:  1139.1364299198722
MAE for train:  425.72959949370363
RMSE for test:  4031.5751432137354
RMSE for train:  1587.0271662266246
r2 for test:  0.8992652658449543
r2 for train:  0.9845908879794064


In [20]:
comparision = []

trees = 100
depth = [7,10,15,20,25]

for dp in depth:
    new_pipe = Pipeline(
        [
            ("preprocessor", transformer),
            ("model", RandomForestRegressor(
                n_estimators=trees,
                max_depth=dp,
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    new_pipe.fit(x_train, y_train)

    y_rf_train = new_pipe.predict(x_train)
    y_rf_test = new_pipe.predict(x_test)

    comparision.append(
        {
            "no. of trees" : trees,
            "max depth" : dp,
            "r2 train" : r2_score(y_train, y_rf_train),
            "r2_test" : r2_score(y_test, y_rf_test),
            "train mae" : mean_absolute_error(y_train, y_rf_train),
            "test mae" : mean_absolute_error(y_test, y_rf_test),
            "train rmse" : np.sqrt(mean_squared_error(y_train, y_rf_train)),
            "test rmse" : np.sqrt(mean_squared_error(y_test, y_rf_test))
        }
    )

    

c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\sarth\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. 

In [21]:
pd.DataFrame(comparision)

,no. of trees,max depth,r2 train,r2_test,train mae,test mae,train rmse,test rmse
0,100,7,0.866436,0.847090,1531.400324,1569.475991,4672.407887,4967.103625
1,100,10,0.911191,0.873974,1348.166327,1437.479498,3809.997562,4509.356924
2,100,15,0.954964,0.889909,1069.537053,1303.734266,2713.173435,4214.643675
3,100,20,0.972624,0.896446,861.372668,1226.474826,2115.331802,4087.603818
4,100,25,0.979798,0.897848,714.068299,1185.079478,1817.178232,4059.846616


NOW FIX THE RANDOM FOREST WITH 200 TREES AND MAX DEPTH AS 20

In [25]:
pipe_rfr = Pipeline([("preprocessor", transformer),
                    ("model",RandomForestRegressor(n_estimators=200,max_depth=20,random_state=42,n_jobs=-1,oob_score=True))])

In [26]:
pipe_rfr.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['year','month','carrier_name','airport_name','arr_flights','peak_delay']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenate

In [27]:
cv_score = cross_val_score(
    pipe_rfr,
    x_train,
    y_train,
    cv=kfold,
    scoring = "r2",
    n_jobs=1
)

In [29]:
print("fold r2 scores: ", cv_score)
print("mean cv r2: ", cv_score.mean())
print("std of fold scores: ", cv_score.std())

fold r2 scores:  [0.88854742 0.87591993 0.88321527 0.88728868 0.88669989]
mean cv r2:  0.88433423692898
std of fold scores:  0.004563520299469723


In [28]:
oob_score = pipe_rfr.named_steps['model'].oob_score_

In [30]:
print(f"oob r2: {oob_score}")

oob r2: 0.8877613434332071


The Random Forest achieved an Out-of-Bag (OOB) score of 0.887, while the mean 5-fold cross-validation R² was 0.885. The close agreement between these two estimates suggests that the model generalizes consistently across different subsets of the data and is not overly dependent on a particular train-test split.

In [21]:
from sklearn.model_selection import GridSearchCV

In [28]:
pipe_rfr = Pipeline([("preprocessor", transformer),
                    ("model",RandomForestRegressor(random_state=42))])

In [32]:
param_grid = {
    'model__n_estimators' : [100,150,200],
    'model__max_depth' : [10,20,25,30],
    'model__max_samples' : [0.60,0.70,0.80]
}

In [33]:
gridsearch = GridSearchCV(
    estimator=pipe_rfr,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

HERE GRIDSEARCHCV WAS NOT USED DUE TO HUGE DATA SIZE AND TOO LONG COMPUTATION TIME

NOW WE MOVE TOWARDS XGBOOST

In [22]:
from xgboost import XGBRegressor

In [23]:
pipe_xgbr = Pipeline(
    [
        ("preprocessor", transformer),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            n_jobs=-1,
            random_state=42
        ))
    ]
)

In [24]:
pipe_xgbr.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['year','month','carrier_name','airport_name','arr_flights','peak_delay']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenate

In [25]:
y_xgbr_train = pipe_xgbr.predict(x_train)
y_xgbr_test = pipe_xgbr.predict(x_test)

In [26]:
mae_test = mean_absolute_error(y_test, y_xgbr_test)
mae_train = mean_absolute_error(y_train, y_xgbr_train)
rmse_test = np.sqrt(mean_squared_error(y_test, y_xgbr_test))
rmse_train = np.sqrt(mean_squared_error(y_train, y_xgbr_train))

r2_test = r2_score(y_test,y_xgbr_test)
r2_train = r2_score(y_train, y_xgbr_train)

In [27]:
print("MAE for test: ",mae_test)
print("MAE for train: ",mae_train)
print("RMSE for test: ",rmse_test)
print("RMSE for train: ",rmse_train)
print("r2 for test: ",r2_test)
print("r2 for train: ",r2_train)

MAE for test:  1224.3169999537222
MAE for train:  1157.5483848888955
RMSE for test:  3862.123182280114
RMSE for train:  3359.4916245835852
r2 for test:  0.907555310133328
r2 for train:  0.93095121898903


In [31]:
cv = cross_val_score(
    pipe_xgbr,
    x_train,
    y_train,
    cv=kfold,
    scoring="r2",
    n_jobs=-1
)

In [32]:
print("fold r2 scores: ", cv)
print("mean cv r2: ", cv.mean())
print("std of fold scores: ", cv.std())

fold r2 scores:  [0.90412048 0.88979876 0.88377584 0.89745133 0.90120998]
mean cv r2:  0.8952712774919676
std of fold scores:  0.0074889631345545875
